# Visualization
Paper-quality figures for the robust CFLP study.  
Edit the **Configuration** cell, then run the figure cells you need.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Path to the Excel file produced by 05_robustification_and_adaptive_pricing.ipynb
EXCEL = "05_sensitivity_results.xlsx"   # relative to experiments/, or use an absolute path

# Default slice for the violin plot
V_SCALE = 0.75
W_VAL   = 10
GAMMAS  = [1, 2, 3, 4]

In [ ]:
import importlib, sys, pathlib

# Make sure experiments/ is on the path so we can import fig_violin_cvar
_here = pathlib.Path().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import fig_violin_cvar as fvc
importlib.reload(fvc)   # picks up any edits without restarting the kernel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams.update(fvc.mpl.rcParams)   # reuse the same style

In [ ]:
# ── Load OOS data ─────────────────────────────────────────────────────────────
oos = pd.read_excel(EXCEL, sheet_name="OOS_Raw")
print(f"Loaded {len(oos)} rows  |  columns: {list(oos.columns)}")

## Fig 1 — Violin + CVaR 5%: Nominal vs Robust
One pair of violins per Γ value. The dark shaded tail is the bottom-5% of each
distribution; the bold horizontal bar marks the CVaR₅% value.

In [ ]:
HALF_W  = 0.32   # half-width of each violin
GAP     = 0.08   # gap between the two violins in a pair
GROUP_W = 2.0    # x-distance between Γ groups

sub = oos[(oos["v_scale"] == V_SCALE) & (oos["w"] == W_VAL)]

fig, ax = plt.subplots(figsize=(10, 5.5))

x_ticks, x_labels = [], []
cvar_summary = {}

for gi, gam in enumerate(GAMMAS):
    g = sub[sub["gamma"] == gam]
    arr_nom = g["profit_a_nom"].values
    arr_rob = g["profit_a_rob"].values

    x_base = gi * GROUP_W
    x_nom  = x_base - (HALF_W + GAP / 2)
    x_rob  = x_base + (HALF_W + GAP / 2)

    lbl_nom = "Nominal $x$" if gi == 0 else None
    lbl_rob = "Robust $x$"  if gi == 0 else None

    cv_nom = fvc.draw_violin(ax, arr_nom, x_nom, HALF_W,
                             fvc.NOM_LIGHT, fvc.NOM_DARK, label=lbl_nom)
    cv_rob = fvc.draw_violin(ax, arr_rob, x_rob, HALF_W,
                             fvc.ROB_LIGHT, fvc.ROB_DARK, label=lbl_rob)

    cvar_summary[gam] = (cv_nom, cv_rob)
    x_ticks.append(x_base)
    x_labels.append(f"$\\Gamma = {gam}$")

ax.axhline(0, color="black", lw=0.9, linestyle=":", zorder=1, alpha=0.6)
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_xlim(x_ticks[0] - GROUP_W * 0.7, x_ticks[-1] + GROUP_W * 0.7)
ax.set_ylabel("Out-of-sample profit")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.grid(axis="y", linestyle="--")

legend_patches = [
    mpatches.Patch(color=fvc.NOM_LIGHT, alpha=0.7, label="Nominal $x$  (Scen. A)"),
    mpatches.Patch(color=fvc.ROB_LIGHT, alpha=0.7, label="Robust $x$   (Scen. A)"),
    mpatches.Patch(color=fvc.NOM_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Nominal"),
    mpatches.Patch(color=fvc.ROB_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Robust"),
    plt.Line2D([0],[0], color="gray", lw=2.2, label="Median"),
    plt.Line2D([0],[0], color="gray", lw=1.2, linestyle="--", label="5th percentile (VaR)"),
    plt.Line2D([0],[0], marker="D", color="w", markeredgecolor="gray", markersize=5, label="Mean"),
]
ax.legend(handles=legend_patches, ncol=2, frameon=True, framealpha=0.92,
          loc="upper right", fontsize=8.5, edgecolor="0.8")
ax.set_title(
    f"OOS Profit Distributions — Nominal vs. Robust  "
    f"($v = {V_SCALE},\\; w = {W_VAL}$, adaptive pricing)",
    fontsize=11, pad=8)

fig.tight_layout()
fig.savefig("fig_violin_cvar.pdf", bbox_inches="tight")
fig.savefig("fig_violin_cvar.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n{'Γ':>4}  {'CVaR5 Nom':>12}  {'CVaR5 Rob':>12}  {'Δ CVaR5':>12}")
print("-" * 46)
for gam, (cn, cr) in cvar_summary.items():
    print(f"{gam:>4}  {cn:>12,.0f}  {cr:>12,.0f}  {cr-cn:>+12,.0f}")